# 04 — PIO–Vehicle Matching and Exploratory Penetration / PMV

This notebook tests whether PIO sales can be matched reliably to cleaned Wholesale/Fleet vehicle volume at the model-month level. It supports exploratory penetration/PMV feasibility—not final KPI reporting, forecasting, dashboards, or data export.

> **Data safety:** Clear all outputs before committing.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pio_vehicle_matching import (
    DEFAULT_MODEL_ALIASES,
    MATCH_KEYS,
    aggregate_pio_model_month,
    aggregate_pio_model_part_month,
    aggregate_vehicle_model_month,
    kpi_outlier_candidates,
    load_matching_inputs,
    match_pio_parts_to_vehicle,
    match_pio_to_vehicle,
    matching_diagnostics,
    model_code_variant_diagnostics,
    model_kpi_summary,
    monthly_pio_vehicle_summary,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load, aggregate, and match

**Question:** Can the two sources be aligned at a stable model-month grain without creating duplicate matches?

PIO uses the step 02 loader; vehicle volume uses the cleaned step 03 logic. Matching remains at model-month level using `month + tentative_brand_group + normalized_model`.

### Data flow

`PIO raw sales` → `PIO model-month / model-part-month` → `Vehicle model-month` → match on `month + tentative report group + normalized model` → calculate exploratory ratios only when matching vehicle volume is available.

In plain English, the notebook first puts both sources at comparable monthly model grains, then checks whether each PIO sales row has vehicle volume that can be used in an exploratory ratio.

In [ ]:
inputs = load_matching_inputs(PROJECT_ROOT / "data" / "raw")
pio_model_month = aggregate_pio_model_month(inputs["pio_analysis"])
pio_model_part_month = aggregate_pio_model_part_month(inputs["pio_analysis"])
vehicle_model_month = aggregate_vehicle_model_month(inputs["vehicle_model_rows"])

matched_model_month, uniqueness = match_pio_to_vehicle(
    pio_model_month, vehicle_model_month
)
diagnostics = matching_diagnostics(matched_model_month, uniqueness)
matched_model_part = match_pio_parts_to_vehicle(
    pio_model_part_month, vehicle_model_month
)

coverage = diagnostics["coverage_summary"].iloc[0]
print(f"Workbook: {inputs['workbook_path'].name}")
print(f"PIO model-month rows: {len(pio_model_month):,}")
print(f"PIO model-part-month rows: {len(pio_model_part_month):,}")
print(f"Vehicle model-month rows: {len(vehicle_model_month):,}")

## Executive summary

This summary deliberately separates three questions:

1. **Key-match coverage:** how many PIO model-month keys find a vehicle record?
2. **Business-value coverage:** how much PIO quantity and revenue are represented by those matches?
3. **Usable vehicle-volume coverage:** how much PIO activity has matching positive vehicle volume and can actually be included in an exploratory ratio?

These are different tests. A successful key match does not by itself validate the model variant or the business denominator.

In [ ]:
key_match_summary = pd.DataFrame([
    {
        "Coverage measure": "PIO model-month keys",
        "Covered / total": f"{int(coverage['matched_model_month_rows']):,} / {int(coverage['pio_model_month_rows']):,}",
        "Coverage": f"{coverage['matched_row_share']:.2%}",
    },
    {
        "Coverage measure": "PIO installed quantity",
        "Covered / total": "Matched keys / all PIO",
        "Coverage": f"{coverage['matched_quantity_share']:.4%}",
    },
    {
        "Coverage measure": "PIO revenue",
        "Covered / total": "Matched keys / all PIO",
        "Coverage": f"{coverage['matched_revenue_share']:.4%}",
    },
])

denominator_summary = diagnostics["coverage_table"].loc[
    diagnostics["coverage_table"]["coverage_type"].ne("Key match")
].copy()
denominator_summary["Coverage"] = denominator_summary["coverage_share"].map(
    lambda value: f"{value:.2%}"
)
denominator_summary["Vehicle-volume basis"] = denominator_summary["coverage_type"].replace({
    "Positive Wholesale denominator": "Usable Wholesale volume",
    "Positive exploratory total denominator": "Usable exploratory total volume",
})
denominator_summary = denominator_summary.rename(columns={
    "basis": "PIO measure",
    "covered": "Covered PIO amount",
    "total": "Total PIO amount",
})

display(Markdown("### A. Key-match and matched-value coverage"))
display(key_match_summary)
display(Markdown("### B. PIO sales with usable vehicle-volume denominator"))
display(denominator_summary[
    ["Vehicle-volume basis", "PIO measure", "Covered PIO amount", "Total PIO amount", "Coverage"]
])

**Takeaway:** PIO and vehicle data match very well by model-month, but we still need to confirm how much PIO sales can actually be divided by vehicle volume and how model variants should be grouped. Strong key matching supports feasibility; it does not make the resulting ratios final business KPIs.

## 2. Matching keys and compact examples

**Question:** What does each analysis grain represent, and how are brand/model keys constructed?

`tentative_brand_group` is a temporary analytical assumption used to align PIO rows with vehicle report groups; it is not an official brand definition. K is tentatively mapped to KUS. In this source, raw `pio_brand_code = H` may include Genesis-like models. For the exploratory match, names such as G80, GV70, and GV80 are temporarily assigned to GMA, while other H models are assigned to HMA. Mobis must confirm the official meaning of H and the correct report-group mapping.

`PIS_SERI` appears to align closely with the vehicle `Model Code`, but it is supporting evidence only. Code-only matching is risky because one code can be associated with base, EV, HEV, PHEV, N, or Coupe variants. The current merge therefore remains exact-name based. Only the explicit `GV60 → GV60 EV` alias is applied; no fuzzy matching is used.

In [ ]:
pio_model_month_display = pio_model_month[
    ["month", "pio_brand_code", "tentative_brand_group", "model", "pio_quantity", "pio_revenue"]
].rename(columns={
    "month": "Month", "pio_brand_code": "Raw PIO Brand Code",
    "tentative_brand_group": "Tentative Report Group", "model": "PIO Model",
    "pio_quantity": "PIO Quantity", "pio_revenue": "PIO Revenue",
})
pio_model_part_display = pio_model_part_month[
    ["month", "tentative_brand_group", "model", "part_number", "quantity", "revenue"]
].rename(columns={
    "month": "Month", "tentative_brand_group": "Tentative Report Group",
    "model": "PIO Model", "part_number": "Part Number",
    "quantity": "PIO Quantity", "revenue": "PIO Revenue",
})
vehicle_model_month_display = vehicle_model_month[
    ["month", "tentative_brand_group", "vehicle_model", "vehicle_model_code",
     "wholesale_volume", "fleet_volume", "total_vehicle_volume"]
].rename(columns={
    "month": "Month", "tentative_brand_group": "Vehicle Report Group",
    "vehicle_model": "Vehicle Model", "vehicle_model_code": "Vehicle Model Code",
    "wholesale_volume": "Wholesale Volume", "fleet_volume": "Fleet Volume",
    "total_vehicle_volume": "Exploratory Total Volume",
})

for title, table in [
    ("PIO model-month example", pio_model_month_display),
    ("PIO model-part-month example", pio_model_part_display),
    ("Vehicle model-month example", vehicle_model_month_display),
]:
    display(Markdown(f"### {title}"))
    display(table.head(5))

display(Markdown("### Tentative mapping example"))
display(
    pio_model_month[
        ["pio_brand_code", "model", "normalized_model", "tentative_brand_group"]
    ].drop_duplicates().sort_values(["tentative_brand_group", "model"]).head(5).rename(columns={
        "pio_brand_code": "Raw PIO Brand Code", "model": "PIO Model",
        "normalized_model": "Normalized Match Name",
        "tentative_brand_group": "Tentative Report Group",
    })
)
print("Explicit model aliases:", DEFAULT_MODEL_ALIASES)

**Takeaway:** The notebook uses readable model names plus a tentative report-group mapping for the current exploratory merge. Neither `pio_brand_code` nor `PIS_SERI` is treated as the official standalone brand/model key.

## 3. Model Code / Variant Mapping Diagnostics

**Question:** Does `PIS_SERI` reinforce the name-based mapping, and where do vehicle variants make a code-only match ambiguous?

`PIS_SERI` is compared with the vehicle report's `Model Code` as supporting evidence. The Sportage example is especially useful: PIO Sportage appears with series values 4 and R, while vehicle code R is associated with Sportage PHEV. This suggests a real relationship between the codes, but also shows why the base name and PHEV variant cannot be treated as interchangeable without sponsor guidance.

In [ ]:
model_code_diagnostics = model_code_variant_diagnostics(
    inputs["pio_raw"], inputs["vehicle_model_rows"]
)

display(Markdown("### PIO models with multiple PIS_SERI values"))
display(model_code_diagnostics["pio_models_multiple_series"].rename(columns={
    "pio_brand_code": "Raw PIO Brand Code", "tentative_brand_group": "Tentative Report Group",
    "pio_model": "PIO Model", "pio_series_count": "PIS_SERI Count",
    "pio_series_values": "PIS_SERI Values",
}))

display(Markdown("### Vehicle model codes linked to multiple model names"))
display(model_code_diagnostics["vehicle_codes_multiple_models"].head(10).rename(columns={
    "tentative_brand_group": "Vehicle Report Group", "vehicle_model_code": "Vehicle Model Code",
    "vehicle_model_count": "Model Name Count", "vehicle_models": "Vehicle Model Names",
}))

display(Markdown("### Sportage code / variant example"))
sportage_variants = model_code_diagnostics["variant_candidates"].loc[
    model_code_diagnostics["variant_candidates"]["pio_model"]
    .astype("string").str.contains("Sportage", case=False, na=False)
]
display(sportage_variants.rename(columns={
    "pio_brand_code": "Raw PIO Brand Code", "tentative_brand_group": "Tentative Report Group",
    "pio_model": "PIO Model", "pio_series_code": "PIS_SERI",
    "vehicle_model": "Vehicle Model", "vehicle_model_code": "Vehicle Model Code",
    "vehicle_name_has_variant_label": "Vehicle Name Has Variant Label",
}))

display(Markdown("### Other likely variant-name candidates"))
display(
    model_code_diagnostics["variant_candidates"].loc[
        model_code_diagnostics["variant_candidates"]
        ["vehicle_name_has_variant_label"]
    ].head(10).rename(columns={
        "pio_brand_code": "Raw PIO Brand Code", "tentative_brand_group": "Tentative Report Group",
        "pio_model": "PIO Model", "pio_series_code": "PIS_SERI",
        "vehicle_model": "Vehicle Model", "vehicle_model_code": "Vehicle Model Code",
        "vehicle_name_has_variant_label": "Vehicle Name Has Variant Label",
    })
)

**Takeaway:** `PIS_SERI` and vehicle model code are promising validation fields, but shared and variant-specific codes prevent code-only matching. Base, EV, HEV, PHEV, N, and Coupe treatment needs Mobis confirmation before final KPI reporting.

## 4. Match diagnostics

**Question:** Are matched keys unique, how much PIO business is covered, and how much PIO sales can actually be divided by vehicle volume?

Coverage is separated into key matching, PIO quantity/revenue represented by those keys, and PIO sales with usable matching vehicle volume. The duplicate-key check matters because a many-to-many merge could silently inflate both PIO and vehicle totals.

In [ ]:
coverage_table = diagnostics["coverage_table"].copy()
coverage_table["coverage_percent"] = coverage_table["coverage_share"] * 100
coverage_table_display = coverage_table[
    ["coverage_type", "basis", "covered", "total", "coverage_percent"]
].rename(columns={
    "coverage_type": "Coverage Category", "basis": "Measure",
    "covered": "Covered", "total": "Total", "coverage_percent": "Coverage (%)",
})
coverage_table_display["Coverage Category"] = coverage_table_display["Coverage Category"].replace({
    "Key match": "Matched model-month key",
    "Positive Wholesale denominator": "PIO sales with usable Wholesale volume",
    "Positive exploratory total denominator": "PIO sales with usable exploratory total volume",
})
display(coverage_table_display)

display(Markdown("### Duplicate / many-to-many check"))
display(diagnostics["key_uniqueness"].rename(columns={
    "dataset": "Dataset", "row_count": "Rows",
    "duplicate_key_rows": "Duplicate-Key Rows",
    "duplicate_key_count": "Duplicate Keys",
    "many_to_many_warning": "Many-to-Many Warning",
}))
print("Model-part merge status:")
print(matched_model_part["_merge"].value_counts(dropna=False))

**Takeaway:** Key-match coverage and business-value coverage are strong, and the uniqueness checks protect against merge inflation. The narrower feasibility test is how much PIO sales has usable matching vehicle volume.

### Month and model coverage summaries

**Question:** Are matching success and usable vehicle-volume coverage consistent across time and models, or concentrated in only a few groups?

The compact distributions summarize all groups. The five lowest-coverage months and models identify where mapping or denominator review should start.

In [ ]:
month_coverage = diagnostics["coverage_by_month"]
model_coverage = diagnostics["coverage_by_model"]
coverage_columns = [
    "matched_row_share", "matched_quantity_share", "matched_revenue_share",
    "valid_wholesale_quantity_share", "valid_total_quantity_share",
]

coverage_labels = {
    "matched_row_share": "Matched key rows",
    "matched_quantity_share": "Matched PIO quantity",
    "matched_revenue_share": "Matched PIO revenue",
    "valid_wholesale_quantity_share": "PIO quantity with usable Wholesale volume",
    "valid_total_quantity_share": "PIO quantity with usable exploratory total volume",
}
display(Markdown("**Month coverage distribution**"))
month_distribution = month_coverage[coverage_columns].describe().T[
    ["count", "mean", "min", "50%", "max"]
].rename(index=coverage_labels, columns={"count": "Groups", "mean": "Mean", "min": "Minimum", "50%": "Median", "max": "Maximum"})
display(month_distribution)
display(Markdown("**Five lowest total-denominator month examples**"))
display(month_coverage.nsmallest(5, "valid_total_quantity_share")[
    ["month", "pio_quantity", "matched_quantity_share", "valid_wholesale_quantity_share", "valid_total_quantity_share"]
].rename(columns={
    "month": "Month", "pio_quantity": "PIO Quantity",
    "matched_quantity_share": "Matched Quantity Share",
    "valid_wholesale_quantity_share": "PIO Quantity Share with Usable Wholesale Volume",
    "valid_total_quantity_share": "PIO Quantity Share with Usable Total Volume",
}))

display(Markdown("**Model coverage distribution**"))
model_distribution = model_coverage[coverage_columns].describe().T[
    ["count", "mean", "min", "50%", "max"]
].rename(index=coverage_labels, columns={"count": "Groups", "mean": "Mean", "min": "Minimum", "50%": "Median", "max": "Maximum"})
display(model_distribution)
display(Markdown("**Five lowest key-match model examples**"))
display(model_coverage.nsmallest(5, "matched_revenue_share")[
    ["tentative_brand_group", "model", "pio_quantity", "pio_revenue", "matched_row_share", "matched_revenue_share"]
].rename(columns={
    "tentative_brand_group": "Tentative Report Group", "model": "PIO Model",
    "pio_quantity": "PIO Quantity", "pio_revenue": "PIO Revenue",
    "matched_row_share": "Matched Row Share", "matched_revenue_share": "Matched Revenue Share",
}))

**Takeaway:** Overall coverage can hide month- or model-specific gaps. The lowest-coverage examples are review queues, not automatic evidence that a PIO record is wrong.

## 5. Unmatched records

**Question:** Which records fail the current exact-name match, and are vehicle-only rows true mapping gaps or simply outside the PIO observation window?

An **unmatched PIO row** means PIO sales exist for that model-month, but the vehicle table does not contain a matching vehicle-volume row under the current match key. Penetration and PMV therefore cannot be calculated for that row.

The current unmatched PIO examples are very small Rio and Stinger rows, so their quantity and revenue impact is negligible. They still need business review to determine whether the cause is discontinued models, naming differences, missing vehicle data, or another reporting rule. Vehicle-only rows are split into the PIO observation window versus months outside that range so future months are not mistaken for matching failures.

In [ ]:
display(Markdown("### Unmatched PIO summary by model"))
display(diagnostics["unmatched_pio_by_model"].rename(columns={
    "pio_brand_code": "Raw PIO Brand Code", "tentative_brand_group": "Tentative Report Group",
    "model": "PIO Model", "normalized_model": "Normalized Match Name",
    "unmatched_months": "Unmatched Months", "pio_quantity": "PIO Quantity",
    "pio_revenue": "PIO Revenue",
}))
display(Markdown("**Unmatched PIO row examples**"))
display(diagnostics["unmatched_pio_model_months"].head(5).rename(columns={
    "month": "Month", "pio_brand_code": "Raw PIO Brand Code",
    "tentative_brand_group": "Tentative Report Group", "model": "PIO Model",
    "pio_quantity": "PIO Quantity", "pio_revenue": "PIO Revenue",
}))

display(Markdown("### Unmatched vehicle date-range summary"))
display(diagnostics["unmatched_vehicle_range_summary"].rename(columns={
    "date_range_group": "Date-Range Group", "row_count": "Rows",
    "unique_models": "Unique Models", "first_month": "First Month", "last_month": "Last Month",
}))
display(Markdown("**Within PIO date range—five examples**"))
display(diagnostics["unmatched_vehicle_within_pio_range"].head(5)[
    ["month", "tentative_brand_group", "vehicle_model", "wholesale_volume", "fleet_volume"]
].rename(columns={
    "month": "Month", "tentative_brand_group": "Vehicle Report Group",
    "vehicle_model": "Vehicle Model", "wholesale_volume": "Wholesale Volume",
    "fleet_volume": "Fleet Volume",
}))
display(Markdown("**Outside PIO date range / future—five examples**"))
display(diagnostics["unmatched_vehicle_outside_pio_range"].head(5)[
    ["month", "tentative_brand_group", "vehicle_model", "wholesale_volume", "fleet_volume"]
].rename(columns={
    "month": "Month", "tentative_brand_group": "Vehicle Report Group",
    "vehicle_model": "Vehicle Model", "wholesale_volume": "Wholesale Volume",
    "fleet_volume": "Fleet Volume",
}))

**Takeaway:** The unmatched Rio/Stinger PIO rows have negligible business-value impact, but they still document a mapping gap. Vehicle-only future months do not reduce PIO coverage; vehicle variants inside the PIO window remain the more important review population.

## 6. Exploratory penetration and PMV definitions

**Question:** Where positive vehicle-volume denominators exist, what exploratory accessory-units-per-vehicle and revenue-per-vehicle patterns appear?

Because PIO quantity appears to count **accessory units rather than unique vehicles**, both penetration fields should be read as accessory-units-per-vehicle proxies—not confirmed penetration rates:

- `penetration_wholesale = PIO accessory units / Wholesale vehicles`: accessory units per matched wholesale vehicle.
- `penetration_total = PIO accessory units / exploratory (Wholesale + Fleet) vehicles`: accessory units per matched exploratory total vehicle.
- Either proxy can exceed 1 because one vehicle may receive multiple PIO accessories.
- `PMVW = PIO revenue / Wholesale vehicles`: PIO revenue per matched wholesale vehicle.
- `PMV_total = PIO revenue / exploratory total vehicle volume`: PIO revenue per matched exploratory Wholesale + Fleet vehicle.

Ratios are populated only when matching vehicle volume is positive. In the ranking tables, an **eligible model** only means that the model has at least 1,000 matched positive exploratory total vehicle units across its matched months, which reduces unstable ratios caused by tiny vehicle volumes. It does **not** mean the model is officially approved, currently sold, or otherwise business-eligible. The threshold is an analytical screen, not a sponsor-approved rule.

In [ ]:
monthly_comparison = monthly_pio_vehicle_summary(
    pio_model_month, vehicle_model_month
)
model_kpis = model_kpi_summary(matched_model_month)
ranking_eligible = model_kpis.loc[
    model_kpis["total_vehicle_volume"] >= 1_000
].copy()
penetration_outliers = kpi_outlier_candidates(
    ranking_eligible, "penetration_total"
)

monthly_comparison_display = monthly_comparison[
    ["month", "pio_quantity", "pio_revenue", "wholesale_volume", "fleet_volume", "total_vehicle_volume"]
].rename(columns={
    "month": "Month", "pio_quantity": "PIO Quantity", "pio_revenue": "PIO Revenue",
    "wholesale_volume": "Wholesale Volume", "fleet_volume": "Fleet Volume",
    "total_vehicle_volume": "Exploratory Total Volume",
})
model_kpi_columns = [
    "tentative_brand_group", "model", "matched_months", "pio_quantity", "pio_revenue",
    "wholesale_volume", "total_vehicle_volume", "penetration_wholesale",
    "penetration_total", "PMVW", "PMV_total",
]
model_kpi_labels = {
    "tentative_brand_group": "Tentative Report Group", "model": "PIO Model",
    "matched_months": "Matched Months", "pio_quantity": "PIO Quantity",
    "pio_revenue": "PIO Revenue", "wholesale_volume": "Wholesale Volume",
    "total_vehicle_volume": "Exploratory Total Volume",
    "penetration_wholesale": "Wholesale Accessory Units / Vehicle",
    "penetration_total": "Total Accessory Units / Vehicle",
    "PMVW": "PIO Revenue / Wholesale Vehicle",
    "PMV_total": "PIO Revenue / Total Vehicle",
}

display(Markdown("### Monthly comparison—five examples"))
display(monthly_comparison_display.head(5))
display(Markdown("### Top five models by PIO revenue"))
display(model_kpis.nlargest(5, "pio_revenue")[model_kpi_columns].rename(columns=model_kpi_labels))
display(Markdown("### Top five eligible models by penetration_total"))
display(ranking_eligible.nlargest(5, "penetration_total")[model_kpi_columns].rename(columns=model_kpi_labels))
display(Markdown("### Top five eligible models by PMV_total"))
display(ranking_eligible.nlargest(5, "PMV_total")[model_kpi_columns].rename(columns=model_kpi_labels))
display(Markdown("### Penetration outlier candidates"))
outlier_display_columns = model_kpi_columns + ["outlier_direction"]
display(penetration_outliers.head(5)[outlier_display_columns].rename(columns={
    **model_kpi_labels,
    "outlier_direction": "Outlier Direction",
}))

**Takeaway:** These rankings are exploratory ratios based on the current exact model-name match. They help identify patterns and review candidates, but variant grouping, channel choice, blank-volume treatment, and the final vehicle-volume definition must be confirmed before business use.

## 7. Exploratory plots


**Question:** Do PIO quantity/revenue and vehicle volume move together over time, and which models with enough usable vehicle volume show the highest exploratory accessory-units-per-vehicle ratios?

The line charts compare direction and timing rather than claiming causality. The model chart uses the same 1,000-unit eligibility screen described above.

In [ ]:
plot_monthly = monthly_comparison.loc[
    monthly_comparison["pio_quantity"].notna()
].copy()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, pio_column, title, color in [
    (axes[0], "pio_revenue", "Monthly PIO Revenue vs Vehicle Volume", "#D97706"),
    (axes[1], "pio_quantity", "Monthly PIO Quantity vs Vehicle Volume", "#2E7D32"),
]:
    vehicle_axis = ax.twinx()
    ax.plot(plot_monthly["month"], plot_monthly[pio_column], marker="o", color=color)
    vehicle_axis.plot(
        plot_monthly["month"], plot_monthly["total_vehicle_volume"],
        marker="s", color="#0066A1",
    )
    ax.set_title(title)
    ax.set_ylabel(pio_column.replace("_", " "))
    vehicle_axis.set_ylabel("exploratory vehicle volume")
axes[1].set_xlabel("Month")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

top_penetration = ranking_eligible.nlargest(10, "penetration_total").sort_values("penetration_total")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_penetration["model"].astype("string"), top_penetration["penetration_total"], color="#7E57C2")
ax.set_title("Top Eligible Models by Exploratory Total Penetration")
ax.set_xlabel("accessory units per exploratory total vehicle")
fig.tight_layout()
plt.show()

**Takeaway:** Visual co-movement supports further feasibility work, but differences in scale, reporting timing, model variants, and denominator completeness prevent causal or final-KPI interpretation.

## 8. Validation


These checks confirm that the exploratory merge does not create a many-to-many row explosion and that each reported KPI uses a positive denominator. Passing them establishes technical consistency, not final business approval of the mapping or denominator.

In [ ]:
assert not uniqueness["many_to_many_warning"].any()
assert not pio_model_month.duplicated(MATCH_KEYS).any()
assert not vehicle_model_month.duplicated(MATCH_KEYS).any()
assert len(matched_model_part) == len(pio_model_part_month)
assert len(matched_model_month) <= len(pio_model_month) + len(vehicle_model_month)
assert matched_model_month.loc[
    matched_model_month["penetration_wholesale"].notna(), "wholesale_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["penetration_total"].notna(), "total_vehicle_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["PMVW"].notna(), "wholesale_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["PMV_total"].notna(), "total_vehicle_volume"
].gt(0).all()
print("All matching and denominator checks passed.")

## 9. Conclusions

This notebook supports **exploratory penetration / PMV feasibility**. It does not produce sponsor-approved penetration, PMV, or PMVW reporting.

### Working assumptions

- `tentative_brand_group` is a temporary analytical assumption used to align PIO rows with vehicle report groups; it is not an official brand definition. K is mapped to KUS; H models with Genesis-like names are mapped to GMA; other H models are mapped to HMA.
- Raw `pio_brand_code = H` may include Genesis-like models in this source. A model such as G80, GV70, or GV80 may therefore be temporarily assigned to GMA for matching, but Mobis needs to confirm the official meaning and mapping.
- `GV60 → GV60 EV` is the only explicit model-name alias. Current matching uses month, normalized model name, and tentative report group.
- `PIS_SERI` appears to align with vehicle `Model Code`, but it is supporting evidence rather than the only key because codes may span base, EV, HEV, PHEV, N, or Coupe variants.
- Exploratory total volume is used only when both Wholesale and Fleet values are available at model-month level.

### Key findings

- PIO and vehicle data match very well by model-month, but how much PIO sales can actually be divided by vehicle volume—and how variants should be grouped—still needs confirmation.
- Key-match coverage, matched quantity/revenue coverage, and how much PIO sales can actually be divided by vehicle volume answer different business questions and should not be combined into one success rate.
- PIO `Rio` and `Stinger` remain unmatched in the current vehicle report. Their quantity/revenue impact is negligible, but they still need business review. Vehicle-only rows include both variant/name gaps inside the PIO window and future months outside it.
- Sportage illustrates the variant risk: PIO uses series values 4 and R, while vehicle code R identifies Sportage PHEV.
- The penetration measures are accessory-units-per-vehicle proxies, not unique-vehicle penetration, and can exceed 1 when a vehicle receives multiple accessories.
- High key-match coverage does not finalize model grouping or vehicle-volume choice. These remain exploratory ratios based on the current exact model-name match.

### Questions for Mobis

1. What is the official H/K-to-HMA/GMA/KUS mapping, including Genesis models currently carried under H?
2. Which base/EV/HEV/PHEV/N/Coupe variants should be combined for penetration and PMV reporting?
3. Do blank Wholesale/Fleet cells mean zero, unavailable, incomplete, or future data?
4. Should the final denominator be Wholesale, Fleet, Wholesale + Fleet, or another sponsor-defined measure?
5. Does PIO quantity count accessory units, vehicles receiving accessories, or another operational unit?
6. What are the official definitions of PMV and PMVW, and is the 1,000-unit ranking screen appropriate?

**Stopping point:** The notebook ends at exploratory matching feasibility. It creates no forecasting model, dashboard, CSV, Excel file, or processed dataset. Clear all outputs before committing.